# 10: The shortcut, measured (E5)

How much of a detector's accuracy comes from a dataset shortcut rather than from the image content?

The cleaned dataset (notebook 03) gives every fake image one JPEG compression so that real and fake images carry the same compression traces. Here we build a second copy of the dataset that skips only that step: the same images, the same crops and the same splits, but the fake images stay uncompressed. Real images are camera JPEGs, so in this copy "has JPEG traces" means "real".

We train the M1 detector (ResNet-50, same settings as notebook 04) on the uncleaned copy and test both models on both test sets:

| | Uncleaned test set | Cleaned test set |
|---|---|---|
| **Trained on uncleaned data** | looks excellent | the real test |
| **Trained on cleaned data (M1)** | | E1 result |

GPU runtime needed (about 30 minutes in total).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q datasets

import io
import json
import shutil
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from datasets import Image as HFImage, load_dataset
from PIL import Image
from sklearn.metrics import average_precision_score, log_loss, roc_auc_score
from torch.utils.data import DataLoader, Dataset
from torchvision.models import ResNet50_Weights, resnet50

ROOT = Path("/content/drive/MyDrive/minor-project")
CLEAN = Path("/content/processed")
UNCLEAN = Path("/content/processed_uncleaned")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

## Settings

In [ ]:
RUN_NAME = "e5_m1_uncleaned"   # the new model trained on uncleaned data
CLEAN_RUN = "m1_resnet50"      # the M1 model from notebook 04 (trained on cleaned data)
EPOCHS = 5
BATCH_SIZE = 64
LEARNING_RATE = 1e-4
NUM_WORKERS = 2
SEED = 42
SIZE = 224

torch.manual_seed(SEED)
np.random.seed(SEED)

## Build the uncleaned copy (about 10 minutes, only the first time)

Start from a copy of the cleaned arrays, then overwrite every fake image with its crop *without* the JPEG step. Real images are left untouched, so the two copies differ in exactly one thing.

In [ ]:
def center_crop_aligned(img, size=SIZE):
    width, height = img.size
    left = ((width - size) // 2) // 16 * 16
    top = ((height - size) // 2) // 16 * 16
    return img.crop((left, top, left + size, top + size))


SPLITS = ["train", "val", "test"]
CLEAN.mkdir(exist_ok=True)
UNCLEAN.mkdir(exist_ok=True)
for split in SPLITS:
    for suffix in ["_images.npy", "_meta.csv"]:
        name = split + suffix
        if not (CLEAN / name).exists():
            shutil.copy(ROOT / "processed" / name, CLEAN / name)

DRIVE_UNCLEAN = ROOT / "processed_uncleaned"
if (DRIVE_UNCLEAN / "test_meta.csv").exists():
    for f in sorted(DRIVE_UNCLEAN.iterdir()):
        if not (UNCLEAN / f.name).exists():
            shutil.copy(f, UNCLEAN / f.name)
    print("Uncleaned copy loaded from Drive")
else:
    ds = load_dataset("TheKernel01/Tiny-GenImage", cache_dir="/content/hf_cache")
    parts = {split: ds[split].cast_column("image", HFImage(decode=False)) for split in ds}

    for split in SPLITS:
        meta = pd.read_csv(CLEAN / f"{split}_meta.csv")
        shutil.copy(CLEAN / f"{split}_images.npy", UNCLEAN / f"{split}_images.npy")
        images = np.load(UNCLEAN / f"{split}_images.npy", mmap_mode="r+")
        fake_rows = meta.index[meta["label"] == "fake"]
        for n, i in enumerate(fake_rows):
            row = meta.loc[i]
            raw = parts[row["hf_split"]][int(row["hf_index"])]["image"]["bytes"]
            img = Image.open(io.BytesIO(raw)).convert("RGB")
            images[i] = np.asarray(center_crop_aligned(img))
            if n % 2000 == 0:
                print(f"{split}: {n}/{len(fake_rows)} fake images")
        images.flush()
        del images
        shutil.copy(CLEAN / f"{split}_meta.csv", UNCLEAN / f"{split}_meta.csv")
        print(f"{split}: done")

    DRIVE_UNCLEAN.mkdir(exist_ok=True)
    for f in sorted(UNCLEAN.iterdir()):
        shutil.copy(f, DRIVE_UNCLEAN / f.name)
        print(f"saved {f.name} to Drive")

## Check: what exactly is different?

Real images must be identical in both copies. Fake images differ only by the JPEG step, which changes each pixel value by a few levels out of 255: invisible to us, but a clear signal for a neural network.

In [ ]:
clean_test = np.load(CLEAN / "test_images.npy", mmap_mode="r")
unclean_test = np.load(UNCLEAN / "test_images.npy", mmap_mode="r")
test_meta_check = pd.read_csv(CLEAN / "test_meta.csv")

real_idx = test_meta_check.index[test_meta_check["label"] == "real"][:500]
fake_idx = test_meta_check.index[test_meta_check["label"] == "fake"][:500]
real_same = all(np.array_equal(clean_test[i], unclean_test[i]) for i in real_idx)
fake_change = np.mean([np.abs(clean_test[i].astype(int) - unclean_test[i].astype(int)).mean() for i in fake_idx])
print(f"Real images identical in both copies: {real_same}")
print(f"Average change per pixel in fake images from the JPEG step: {fake_change:.2f} (out of 255)")

i = fake_idx[0]
before = unclean_test[i][:64, :64].astype(int)
after = clean_test[i][:64, :64].astype(int)
difference = np.clip(np.abs(after - before).mean(axis=2) * 10, 0, 255)
fig, axes = plt.subplots(1, 3, figsize=(10, 3.6))
for ax, picture, title in zip(
    axes,
    [before, after, difference],
    ["Fake, no JPEG (uncleaned)", "Same fake after JPEG (cleaned)", "Difference x 10"],
):
    ax.imshow(picture.astype(np.uint8), cmap="gray", vmin=0, vmax=255, interpolation="nearest")
    ax.set_title(title, fontsize=10)
    ax.axis("off")
plt.tight_layout()
plt.savefig(ROOT / "results" / "e5_jpeg_difference.png", dpi=120)
plt.show()

## Data

In [ ]:
def load_split(folder, split):
    images = np.load(folder / f"{split}_images.npy", mmap_mode="r")
    meta = pd.read_csv(folder / f"{split}_meta.csv")
    meta["y"] = (meta["label"] == "fake").astype(int)
    return images, meta


class ImageArrayDataset(Dataset):
    def __init__(self, images, meta):
        self.images = images
        self.rows = meta.index.to_numpy()
        self.labels = meta["y"].to_numpy()

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i):
        return torch.from_numpy(np.array(self.images[self.rows[i]])), self.labels[i]


def make_loader(images, meta, shuffle=False):
    batch = BATCH_SIZE if shuffle else BATCH_SIZE * 2
    dataset = ImageArrayDataset(images, meta)
    return dataset, DataLoader(dataset, batch_size=batch, shuffle=shuffle, num_workers=NUM_WORKERS, pin_memory=True)


train_set, train_loader = make_loader(*load_split(UNCLEAN, "train"), shuffle=True)
val_set, val_loader = make_loader(*load_split(UNCLEAN, "val"))

test_sets = {}
for name, folder in [("uncleaned", UNCLEAN), ("cleaned", CLEAN)]:
    images, meta = load_split(folder, "test")
    test_sets[name] = (meta, make_loader(images, meta)[1])

print(f"train: {len(train_set)} images, val: {len(val_set)}, test: {len(test_sets['cleaned'][0])} (two versions)")

MEAN = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)


def to_model_input(batch, augment=False):
    x = batch.to(device, non_blocking=True).permute(0, 3, 1, 2).float() / 255
    if augment:
        flip = torch.rand(x.shape[0], device=device) < 0.5
        x[flip] = x[flip].flip(-1)
    return (x - MEAN) / STD

## Model (same as M1)

In [ ]:
def new_resnet50(pretrained=True):
    net = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2 if pretrained else None)
    net.fc = nn.Linear(net.fc.in_features, 1)
    return net.to(device)


model = new_resnet50()
loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
use_amp = device.type == "cuda"
scaler = torch.amp.GradScaler(device.type, enabled=use_amp)


@torch.no_grad()
def predict(net, loader):
    net.eval()
    probs = []
    for images, _ in loader:
        with torch.autocast(device.type, enabled=use_amp):
            logits = net(to_model_input(images)).squeeze(1)
        probs.append(torch.sigmoid(logits.float()).cpu())
    return torch.cat(probs).numpy()


def scores(labels, probs):
    return {
        "loss": float(log_loss(labels, probs, labels=[0, 1])),
        "accuracy": float(((probs >= 0.5) == labels).mean()),
        "auc": float(roc_auc_score(labels, probs)),
        "ap": float(average_precision_score(labels, probs)),
    }

## Train on the uncleaned data

In [ ]:
CKPT_DIR = ROOT / "checkpoints"
CKPT_DIR.mkdir(exist_ok=True)
history = []
best_val_loss = float("inf")

for epoch in range(1, EPOCHS + 1):
    model.train()
    start = time.time()
    running_loss = 0.0
    for images, labels in train_loader:
        x = to_model_input(images, augment=True)
        y = labels.float().to(device)
        with torch.autocast(device.type, enabled=use_amp):
            loss = loss_fn(model(x).squeeze(1), y)
        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        running_loss += loss.item() * len(y)

    val = scores(val_set.labels, predict(model, val_loader))
    history.append({"epoch": epoch, "train_loss": running_loss / len(train_set), **{f"val_{k}": v for k, v in val.items()}})
    print(f"Epoch {epoch}: train loss {history[-1]['train_loss']:.4f} | val loss {val['loss']:.4f} "
          f"| val acc {val['accuracy']:.3f} | val AUC {val['auc']:.4f} | {time.time() - start:.0f}s")

    if val["loss"] < best_val_loss:
        best_val_loss = val["loss"]
        torch.save(model.state_dict(), CKPT_DIR / f"{RUN_NAME}.pt")
        print("  saved as best so far")

## Test: both models on both test sets

The cleaned-data model is the M1 checkpoint from notebook 04, so its cleaned-test numbers should match the E1 result exactly.

In [ ]:
models = {"uncleaned": RUN_NAME, "cleaned": CLEAN_RUN}
rows = []
for train_name, run in models.items():
    net = new_resnet50(pretrained=False)
    net.load_state_dict(torch.load(CKPT_DIR / f"{run}.pt", map_location=device))
    for test_name, (meta, loader) in test_sets.items():
        probs = predict(net, loader)
        out = meta.copy()
        out["prob_fake"] = probs
        out["correct"] = (out["prob_fake"] >= 0.5).astype(int) == out["y"]
        out.to_csv(ROOT / "results" / f"e5_train-{train_name}_test-{test_name}_predictions.csv", index=False)
        row = {"trained_on": train_name, "tested_on": test_name, **scores(out["y"].to_numpy(), probs)}
        row.update(out.groupby("generator")["correct"].mean().to_dict())
        rows.append(row)

summary = pd.DataFrame(rows)
summary.to_csv(ROOT / "results" / "e5_summary.csv", index=False)
with open(ROOT / "results" / f"{RUN_NAME}.json", "w") as f:
    json.dump({"run": RUN_NAME, "epochs": EPOCHS, "history": history, "results": rows}, f, indent=2)

print("Overall:")
display(summary.pivot(index="trained_on", columns="tested_on", values="accuracy").round(3))
print("\nPer source (Real = correctly called real, others = correctly called fake):")
sources = ["Real", "ADM", "GLIDE", "Midjourney", "SD15", "VQDM", "Wukong"]
display(summary.set_index(["trained_on", "tested_on"])[["auc"] + sources].round(3))

## Chart

In [ ]:
COLORS = {"uncleaned": "#eb6834", "cleaned": "#2a78d6"}
LABELS = {"uncleaned": "Trained on uncleaned data", "cleaned": "Trained on cleaned data"}
table = summary.set_index(["trained_on", "tested_on"])

fig, axes = plt.subplots(1, 2, figsize=(13, 4.4), gridspec_kw={"width_ratios": [1, 2.4]})
width = 0.38

x = np.arange(2)
for k, train_name in enumerate(models):
    values = [table.loc[(train_name, t), "accuracy"] * 100 for t in ["uncleaned", "cleaned"]]
    bars = axes[0].bar(x + (k - 0.5) * width, values, width - 0.03, color=COLORS[train_name], label=LABELS[train_name])
    axes[0].bar_label(bars, fmt="%.1f", fontsize=9)
axes[0].set_xticks(x, ["Uncleaned test set", "Cleaned test set"])
axes[0].set_ylabel("accuracy (%)")
axes[0].set_title("Overall accuracy")

x = np.arange(len(sources))
for k, train_name in enumerate(models):
    values = [table.loc[(train_name, "cleaned"), s] * 100 for s in sources]
    bars = axes[1].bar(x + (k - 0.5) * width, values, width - 0.03, color=COLORS[train_name], label=LABELS[train_name])
    axes[1].bar_label(bars, fmt="%.0f", fontsize=8)
axes[1].set_xticks(x, sources)
axes[1].set_ylabel("correct (%)")
axes[1].set_title("Cleaned test set, per source")

for ax in axes:
    ax.set_ylim(0, 110)
    ax.axhline(50, color="#999999", linewidth=1, linestyle="--")
    ax.grid(axis="y", color="#e5e5e5")
    ax.set_axisbelow(True)
    ax.spines[["top", "right"]].set_visible(False)
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=2, frameon=False)
plt.tight_layout(rect=[0, 0, 1, 0.92])
plt.savefig(ROOT / "results" / "e5_shortcut.png", dpi=120)
plt.show()